TUNING WITH DIFFERENT MODELS -AVOIDING OVERFITTING

HOUR-WISE PREDICTIONS

In [71]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [68]:
# Load hour-wise dataset
hour_df = pd.read_csv("/content/hour.csv")

# Drop unnecessary columns
hour_df.drop(['instant','dteday','casual','registered'], axis=1, inplace=True)

# Remove null values
hour_df.dropna(inplace=True)

# Add lag feature during preprocessing (moved here to ensure X_hour includes it)
hour_df["cnt_lag1"] = hour_df["cnt"].shift(1)
hour_df.dropna(inplace=True)

# Features & target
X_hour = hour_df.drop('cnt', axis=1)
y_hour = hour_df['cnt']

# Train-test split
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_hour, y_hour, test_size=0.2, random_state=42
)

DECISION TREE REGRESSION

In [32]:
from sklearn.tree import DecisionTreeRegressor

dt_hour = DecisionTreeRegressor(
    max_depth=6,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

dt_hour.fit(Xh_train, yh_train)

train_pred = dt_hour.predict(Xh_train)
test_pred  = dt_hour.predict(Xh_test)

print("DT HOUR Train R2:", r2_score(yh_train, train_pred))
print("DT HOUR Test  R2:", r2_score(yh_test, test_pred))
print("DT HOUR Test RMSE:", np.sqrt(mean_squared_error(yh_test, test_pred)))
print("DT HOUR Test MAE:", mean_absolute_error(yh_test, test_pred))


DT HOUR Train R2: 0.8735797128886814
DT HOUR Test  R2: 0.7757957434507342
DT HOUR Test RMSE: 948.1733371920574
DT HOUR Test MAE: 679.5055320427431


RANDOM FOREST REGRESSION

In [33]:
from sklearn.ensemble import RandomForestRegressor

rf_hour = RandomForestRegressor(
    n_estimators=120,
    max_depth=8,
    min_samples_split=25,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf_hour.fit(Xh_train, yh_train)

train_pred = rf_hour.predict(Xh_train)
test_pred  = rf_hour.predict(Xh_test)

print("RF HOUR Train R2:", r2_score(yh_train, train_pred))
print("RF HOUR Test  R2:", r2_score(yh_test, test_pred))
print("RF HOUR Test RMSE:", np.sqrt(mean_squared_error(yh_test, test_pred)))
print("RF HOUR Test MAE:", mean_absolute_error(yh_test, test_pred))


RF HOUR Train R2: 0.8862037649928478
RF HOUR Test  R2: 0.8345256711284861
RF HOUR Test RMSE: 814.5750920308811
RF HOUR Test MAE: 523.6677756897082


GRADIENT BOOSTING REGRESSION

In [69]:
from sklearn.ensemble import GradientBoostingRegressor

gb_hour = GradientBoostingRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.75,
    random_state=42
)

gb_hour.fit(Xh_train, yh_train)

# Assign gb_hour as best_hour_model for subsequent use
best_hour_model = gb_hour

train_pred = gb_hour.predict(Xh_train)
test_pred  = gb_hour.predict(Xh_test)

print("GB HOUR Train R2:", r2_score(yh_train, train_pred))
print("GB HOUR Test  R2:", r2_score(yh_test, test_pred))
print("GB HOUR Test RMSE:", np.sqrt(mean_squared_error(yh_test, test_pred)))
print("GB HOUR Test MAE:", mean_absolute_error(yh_test, test_pred))

GB HOUR Train R2: 0.8514616793015874
GB HOUR Test  R2: 0.8530917338480826
GB HOUR Test RMSE: 68.03952799365389
GB HOUR Test MAE: 44.3129252162033


In [40]:

import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


Ridge Regression

In [41]:
ridge_hour = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=10.0))
])

ridge_hour.fit(Xh_train, yh_train)

train_pred = ridge_hour.predict(Xh_train)
test_pred  = ridge_hour.predict(Xh_test)

print("RIDGE HOUR Train R2:", r2_score(yh_train, train_pred))
print("RIDGE HOUR Test  R2:", r2_score(yh_test, test_pred))


RIDGE HOUR Train R2: 0.7908608543080133
RIDGE HOUR Test  R2: 0.8272214761586192


SVR

In [42]:
svr_hour = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(C=10, epsilon=0.1, kernel="rbf"))
])

svr_hour.fit(Xh_train, yh_train)

train_pred = svr_hour.predict(Xh_train)
test_pred  = svr_hour.predict(Xh_test)

print("SVR HOUR Train R2:", r2_score(yh_train, train_pred))
print("SVR HOUR Test  R2:", r2_score(yh_test, test_pred))


SVR HOUR Train R2: 0.23054234695083553
SVR HOUR Test  R2: 0.22163212958429224


Neural Network

In [43]:
scaler_h = StandardScaler()
Xh_train_s = scaler_h.fit_transform(Xh_train)
Xh_test_s  = scaler_h.transform(Xh_test)

nn_hour = Sequential([
    Dense(64, activation="relu", input_shape=(Xh_train_s.shape[1],)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1)
])

nn_hour.compile(optimizer="adam", loss="mse")

nn_hour.fit(
    Xh_train_s, yh_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

train_pred = nn_hour.predict(Xh_train_s).ravel()
test_pred  = nn_hour.predict(Xh_test_s).ravel()

print("NN HOUR Train R2:", r2_score(yh_train, train_pred))
print("NN HOUR Test  R2:", r2_score(yh_test, test_pred))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
NN HOUR Train R2: 0.13717567920684814
NN HOUR Test  R2: 0.2998112440109253


PRETRAINED NN

In [44]:
# Scale data
scaler_hour = StandardScaler()
Xh_train_s = scaler_hour.fit_transform(Xh_train)
Xh_test_s  = scaler_hour.transform(Xh_test)

# Pretrained base
base_hour = Sequential([
    Dense(64, activation="relu", input_shape=(Xh_train_s.shape[1],)),
    Dense(32, activation="relu")
])

base_hour.trainable = False

# Final model
pretrained_hour = Sequential([
    base_hour,
    Dropout(0.3),
    Dense(1)
])

pretrained_hour.compile(optimizer="adam", loss="mse")

pretrained_hour.fit(
    Xh_train_s, yh_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

train_pred = pretrained_hour.predict(Xh_train_s).ravel()
test_pred  = pretrained_hour.predict(Xh_test_s).ravel()

print("PRETRAINED HOUR Train R2:", r2_score(yh_train, train_pred))
print("PRETRAINED HOUR Test  R2:", r2_score(yh_test, test_pred))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
PRETRAINED HOUR Train R2: -5.666344165802002
PRETRAINED HOUR Test  R2: -4.559685707092285


HOUR+6 PREDICTIONS

In [47]:
# ===================== REQUIRED IMPORTS =====================
import numpy as np
import pandas as pd


In [59]:
# Add lag feature during preprocessing
hour_df["cnt_lag1"] = hour_df["cnt"].shift(1)
hour_df.dropna(inplace=True)

In [66]:
hour_feature_cols = [
    "season", "yr", "mnth", "hr", "holiday", "weekday",
    "workingday", "weathersit", "temp",
    "atemp", "hum", "windspeed", "cnt_lag1"
]

In [61]:
def recursive_forecast_hour(model, input_df, steps):
    preds = []
    current = input_df.copy()

    for _ in range(steps):
        pred = model.predict(current)[0]
        preds.append(pred)

        # feed prediction back (KEY FIX)
        current["cnt_lag1"] = pred

        # increment hour
        current["hr"] = (current["hr"].iloc[0] + 1) % 24

        # increment weekday if day changes
        if current["hr"].iloc[0] == 0:
            current["weekday"] = (current["weekday"].iloc[0] + 1) % 7

    return preds


In [62]:
def format_hour(hour):
    if hour == 0:
        return "12 AM"
    elif hour < 12:
        return f"{hour} AM"
    elif hour == 12:
        return "12 PM"
    else:
        return f"{hour-12} PM"


In [63]:
def predict_next_6_hours(model, user_input, feature_cols):
    df = pd.DataFrame([user_input])[feature_cols]

    preds = recursive_forecast_hour(model, df, steps=6)

    start_hour = user_input["hr"]

    hour_labels = [format_hour((start_hour+i) % 24) for i in range(6)]

    result = pd.DataFrame({
        "Time": hour_labels,
        "Predicted Count": np.round(preds, 2)
    })

    result.index = [""] * len(result)
    return result


In [94]:
hour_input = {
    "season": 1,
    "yr": 0,   # Assuming 'yr' corresponds to '0' for the first year in the dataset (2011)
    "mnth": 1,
    "hr": 10,
    "holiday": 0,
    "weekday": 2,
    "workingday": 1,
    "weathersit": 1,
    "temp": 0.24,
    "atemp": 0.2879,
    "hum": 0.81,
    "windspeed": 0.0,
    "cnt_lag1": 85   # last known hour demand
}

In [95]:
print("Hourly Forecast (Next 6 Hours)")
print(
    predict_next_6_hours(
        best_hour_model,
        hour_input,
        hour_feature_cols
    )
)


Hourly Forecast (Next 6 Hours)
   Time  Predicted Count
  10 AM           102.99
  11 AM           136.22
  12 PM           154.06
   1 PM           159.67
   2 PM           171.92
   3 PM           181.60


DAY-WISE PREDICTIONS

In [93]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


In [25]:
# Load day-wise dataset (renamed from hour_df to day_df for clarity)
day_df = pd.read_csv("/content/day (1).csv")

# Drop unnecessary columns
day_df.drop(['instant','dteday','casual','registered'], axis=1, inplace=True)

# Remove null values
day_df.dropna(inplace=True)

# Features & target
X_day = day_df.drop('cnt', axis=1)
y_day = day_df['cnt']

# Train-test split
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_day, y_day, test_size=0.2, random_state=42
)

DECISION TREE REGRESSION

In [29]:
from sklearn.tree import DecisionTreeRegressor

dt_day = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)

dt_day.fit(Xd_train, yd_train)

train_pred = dt_day.predict(Xd_train)
test_pred  = dt_day.predict(Xd_test)

print("DT DAY Train R2:", r2_score(yd_train, train_pred))
print("DT DAY Test  R2:", r2_score(yd_test, test_pred))
print("DT DAY Test RMSE:", np.sqrt(mean_squared_error(yd_test, test_pred)))
print("DT DAY Test MAE:", mean_absolute_error(yd_test, test_pred))


DT DAY Train R2: 0.861982339450825
DT DAY Test  R2: 0.7810053949500475
DT DAY Test RMSE: 937.0926251364559
DT DAY Test MAE: 656.8529527409446


RANDOM FOREST REGRESSION

In [73]:
from sklearn.ensemble import RandomForestRegressor

rf_day = RandomForestRegressor(
    n_estimators=100,
    max_depth=7,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf_day.fit(Xd_train, yd_train)

train_pred = rf_day.predict(Xd_train)
test_pred  = rf_day.predict(Xd_test)

print("RF DAY Train R2:", r2_score(yd_train, train_pred))
print("RF DAY Test  R2:", r2_score(yd_test, test_pred))
print("RF DAY Test RMSE:", np.sqrt(mean_squared_error(yd_test, test_pred)))
print("RF DAY Test MAE:", mean_absolute_error(yd_test, test_pred))

RF DAY Train R2: 0.8915012595398295
RF DAY Test  R2: 0.8399863159146971
RF DAY Test RMSE: 801.0218709527506
RF DAY Test MAE: 517.6398181960329


GRADIENT BOOSTING REGRESSION

In [74]:
from sklearn.ensemble import GradientBoostingRegressor

gb_day = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.8,
    random_state=42
)

gb_day.fit(Xd_train, yd_train)

train_pred = gb_day.predict(Xd_train)
test_pred  = gb_day.predict(Xd_test)

print("GB DAY Train R2:", r2_score(yd_train, train_pred))
print("GB DAY Test  R2:", r2_score(yd_test, test_pred))
print("GB DAY Test RMSE:", np.sqrt(mean_squared_error(yd_test, test_pred)))
print("GB DAY Test MAE:", mean_absolute_error(yd_test, test_pred))

GB DAY Train R2: 0.8926661994240375
GB DAY Test  R2: 0.8652282234464708
GB DAY Test RMSE: 735.1319336030266
GB DAY Test MAE: 534.388510223112


In [76]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Ridge Regression

In [77]:
ridge_day = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=5.0))
])

ridge_day.fit(Xd_train, yd_train)

train_pred = ridge_day.predict(Xd_train)
test_pred  = ridge_day.predict(Xd_test)

print("RIDGE DAY Train R2:", r2_score(yd_train, train_pred))
print("RIDGE DAY Test  R2:", r2_score(yd_test, test_pred))

RIDGE DAY Train R2: 0.791020591327732
RIDGE DAY Test  R2: 0.827513339722902


SVR

In [37]:
svr_day = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(C=10, epsilon=0.2, kernel="rbf"))
])

svr_day.fit(Xd_train, yd_train)

train_pred = svr_day.predict(Xd_train)
test_pred  = svr_day.predict(Xd_test)

print("SVR DAY Train R2:", r2_score(yd_train, train_pred))
print("SVR DAY Test  R2:", r2_score(yd_test, test_pred))


SVR DAY Train R2: 0.2305374806244932
SVR DAY Test  R2: 0.22162836770413974


Neural Network

In [38]:
scaler_d = StandardScaler()
Xd_train_s = scaler_d.fit_transform(Xd_train)
Xd_test_s  = scaler_d.transform(Xd_test)

nn_day = Sequential([
    Dense(64, activation="relu", input_shape=(Xd_train_s.shape[1],)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1)
])

nn_day.compile(optimizer="adam", loss="mse")

nn_day.fit(
    Xd_train_s, yd_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

train_pred = nn_day.predict(Xd_train_s).ravel()
test_pred  = nn_day.predict(Xd_test_s).ravel()

print("NN DAY Train R2:", r2_score(yd_train, train_pred))
print("NN DAY Test  R2:", r2_score(yd_test, test_pred))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
NN DAY Train R2: -0.0404278039932251
NN DAY Test  R2: 0.14446532726287842


Using a pre-trained Dense layer (transfer learning style for tabular data)

In [39]:
# Scale data
scaler_day = StandardScaler()
Xd_train_s = scaler_day.fit_transform(Xd_train)
Xd_test_s  = scaler_day.transform(Xd_test)

# Pretrained base (simulated)
base_day = Sequential([
    Dense(64, activation="relu", input_shape=(Xd_train_s.shape[1],)),
    Dense(32, activation="relu")
])

base_day.trainable = False  # freeze pretrained layers

# Final model
pretrained_day = Sequential([
    base_day,
    Dropout(0.3),
    Dense(1)
])

pretrained_day.compile(optimizer="adam", loss="mse")

pretrained_day.fit(
    Xd_train_s, yd_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

train_pred = pretrained_day.predict(Xd_train_s).ravel()
test_pred  = pretrained_day.predict(Xd_test_s).ravel()

print("PRETRAINED DAY Train R2:", r2_score(yd_train, train_pred))
print("PRETRAINED DAY Test  R2:", r2_score(yd_test, test_pred))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
PRETRAINED DAY Train R2: -5.665796756744385
PRETRAINED DAY Test  R2: -4.559043884277344


DAY+6 PREDICTIONS

In [96]:
day_feature_cols = [
    "season", "mnth", "weekday", "holiday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed", "cnt_lag1"
]


In [97]:
def recursive_forecast_day(model, input_df, steps):
    preds = []
    current = input_df.copy()

    for _ in range(steps):
        pred = model.predict(current)[0]
        preds.append(pred)

        # --- Update lag correctly ---
        current.loc[0, "cnt_lag1"] = pred

        # Increment weekday
        current.loc[0, "weekday"] = (current.loc[0, "weekday"] + 1) % 7

        # Increment month if weekday rolls over
        if current.loc[0, "weekday"] == 0:
            current.loc[0, "mnth"] = (current.loc[0, "mnth"] % 12) + 1

        # Optional: add small variability to temp/hum for different predictions
        current.loc[0, "temp"] += np.random.uniform(-0.01, 0.01)
        current.loc[0, "hum"] += np.random.uniform(-0.01, 0.01)

    return preds


In [98]:
def format_day_label(start_weekday, i):
    weekdays = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
    return weekdays[(start_weekday + i) % 7]


In [99]:
def predict_next_6_days(model, user_input, feature_cols):
    df = pd.DataFrame([user_input])[feature_cols]
    preds = recursive_forecast_day(model, df, steps=6)

    start_weekday = user_input["weekday"]
    day_labels = [format_day_label(start_weekday, i) for i in range(6)]

    result = pd.DataFrame({
        "Day": day_labels,
        "Predicted Count": np.round(preds, 2)
    })
    result.index = [""] * len(result)
    return result


In [ ]:
day_input = {
    "season": 1,
    "mnth": 5,
    "weekday": 5,      # current weekday (0=Sun)
    "holiday": 0,
    "workingday": 1,
    "weathersit": 1,
    "temp": 0.26,
    "atemp": 0.3,
    "hum": 0.78,
    "windspeed": 0.0,
    "cnt_lag1": 150    # last known day count
}


In [100]:
print("Day-wise Forecast (Next 6 Days)")
forecast_day6 = predict_next_6_days(best_day_model, day_input, day_feature_cols)
print(forecast_day6)


Day-wise Forecast (Next 6 Days)
  Day  Predicted Count
  Tue          1291.83
  Wed          1409.80
  Thu          1573.71
  Fri          1553.50
  Sat          1583.28
  Sun          1569.28


/tmp/ipython-input-898193872.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1291.83' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  current.loc[0, "cnt_lag1"] = pred
